# Voltametria Cíclica Reversível — Mecanismo EC

## Bibliotecas:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [ ]:
model = pybamm.BaseModel()

concentration_o = pybamm.Variable("Concentração de O", domain="electrolyte")
concentration_r = pybamm.Variable("Concentração de R", domain="electrolyte")
concentration_p = pybamm.Variable("Concentração de P", domain="electrolyte")

initial_potential    = pybamm.Parameter("Potencial Inicial [V]")
final_potential      = pybamm.Parameter("Potencial Final [V]")
standard_potential   = pybamm.Parameter("Potencial Padrão [V]")
scan_rate            = pybamm.Parameter("Velocidade de Varredura [V.s-1]")
faraday              = pybamm.Parameter("Constante de Faraday [C.mol-1]")
gas_constant         = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
temperature          = pybamm.Parameter("Temperatura [K]")
diffusion_ratio      = pybamm.Parameter("Razão de Difusão")
equilibrium_constant = pybamm.Parameter("Constante de Equilíbrio")

flux_o = -pybamm.grad(concentration_o)   # fluxo difusional de O
flux_r = -pybamm.grad(concentration_r)   # fluxo difusional de R
flux_p = -pybamm.grad(concentration_p)   # fluxo difusional de P

# equilíbrio R ⇌ P assumido instantâneo — sem termos cinéticos no rhs
model.rhs = {
    concentration_o: -pybamm.div(flux_o),
    concentration_r: -pybamm.div(flux_r),
    concentration_p: -diffusion_ratio * pybamm.div(flux_p),
}

# condições iniciais
model.initial_conditions = {
    concentration_o: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    concentration_r: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
    concentration_p: pybamm.Scalar(0),  # c_P(x,0) = 0: P inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): equilíbrio instantâneo R ⇌ P na superfície do eletrodo
# direita  (x=6): seio da solução — difusão semi-infinita
nernst_factor     = faraday / (gas_constant * temperature)           # f = F/(RT)
total_time        = 2 * (final_potential - initial_potential) / scan_rate
cosine            = pybamm.cos((np.pi / total_time) * pybamm.t)
sine              = pybamm.sin((np.pi / total_time) * pybamm.t)
applied_potential = initial_potential + (2 * (final_potential - initial_potential) / np.pi) * pybamm.AbsoluteValue(pybamm.arctan(sine / cosine))
overpotential     = applied_potential - standard_potential
theta             = pybamm.exp(nernst_factor * overpotential)        # θ = exp(f·η)

model.boundary_conditions = {
    concentration_o: {
        "left":  (theta / (1 + theta + equilibrium_constant),                "Dirichlet"),  # equilíbrio de Nernst + EC
        "right": (pybamm.Scalar(1),                                           "Dirichlet"),  # difusão semi-infinita
    },
    concentration_r: {
        "left":  (1 / (1 + theta + equilibrium_constant),                    "Dirichlet"),  # equilíbrio de Nernst + EC
        "right": (pybamm.Scalar(0),                                           "Dirichlet"),  # difusão semi-infinita
    },
    concentration_p: {
        "left":  (equilibrium_constant / (1 + theta + equilibrium_constant), "Dirichlet"),  # equilíbrio instantâneo R ⇌ P
        "right": (pybamm.Scalar(0),                                           "Dirichlet"),  # difusão semi-infinita
    },
}

model.variables = {
    "Concentração de O":  concentration_o,
    "Concentração de R":  concentration_r,
    "Concentração de P":  concentration_p,
    "Fluxo de O":         flux_o,
    "Fluxo de R":         flux_r,
    "Fluxo de P":         flux_p,
    "Potencial Aplicado": applied_potential,
}

param = pybamm.ParameterValues(
    {
        "Potencial Inicial [V]":             "[input]",
        "Potencial Final [V]":               "[input]",
        "Potencial Padrão [V]":              "[input]",
        "Velocidade de Varredura [V.s-1]":   "[input]",
        "Constante de Faraday [C.mol-1]":    96485.3,
        "Constante dos Gases [J.K-1.mol-1]": 8.31446,
        "Temperatura [K]":                   298.15,
        "Razão de Difusão":                  1,
        "Constante de Equilíbrio":           "[input]",
    }
)


# ── Geometria e Malha ─────────────────────────────────────────────────────────

x_variable = pybamm.SpatialVariable(
    "x", domain=["electrolyte"], coord_sys="cartesian"
)

geometry = {
    "electrolyte": {x_variable: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}
}

submesh_types   = {"electrolyte": pybamm.Uniform1DSubMesh}
variable_points = {x_variable: 400}
mesh            = pybamm.Mesh(geometry, submesh_types, variable_points)


# ── Discretização ─────────────────────────────────────────────────────────────

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
discretisation  = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
discretisation.process_model(model)

## Gráfico Estático:

In [ ]:
static_solver = pybamm.ScipySolver()

# potenciais fixos para a varredura cíclica
DEFAULT_INITIAL_POTENTIAL  =  0.3   # potencial inicial [V]
DEFAULT_FINAL_POTENTIAL    = -0.3   # potencial final [V]
DEFAULT_STANDARD_POTENTIAL =  0     # potencial padrão [V]

# ── variando a velocidade de varredura ────────────────────────────────────────
scan_rates            = [-0.05, -0.1, -0.5]
DEFAULT_EQUILIBRIUM   = 10

currents_scan_rate   = []
potentials_scan_rate = []

for scan_rate_i in scan_rates:
    final_time = 2 * (DEFAULT_FINAL_POTENTIAL - DEFAULT_INITIAL_POTENTIAL) / scan_rate_i
    time       = np.linspace(0, final_time, 1000)

    solution = static_solver.solve(
        model, time,
        inputs={
            "Potencial Inicial [V]":           DEFAULT_INITIAL_POTENTIAL,
            "Potencial Final [V]":             DEFAULT_FINAL_POTENTIAL,
            "Potencial Padrão [V]":            DEFAULT_STANDARD_POTENTIAL,
            "Velocidade de Varredura [V.s-1]": scan_rate_i,
            "Constante de Equilíbrio":         DEFAULT_EQUILIBRIUM,
        }
    )
    currents_scan_rate.append(-solution["Fluxo de O"](solution.t, x=0))
    potentials_scan_rate.append(solution["Potencial Aplicado"](solution.t))

# ── variando a constante de equilíbrio ────────────────────────────────────────
equilibrium_constants  = [0.1, 10, 1000]
DEFAULT_SCAN_RATE      = -0.1

currents_equilibrium   = []
potentials_equilibrium = []

for equilibrium_constant_i in equilibrium_constants:
    final_time = 2 * (DEFAULT_FINAL_POTENTIAL - DEFAULT_INITIAL_POTENTIAL) / DEFAULT_SCAN_RATE
    time       = np.linspace(0, final_time, 1000)

    solution = static_solver.solve(
        model, time,
        inputs={
            "Potencial Inicial [V]":           DEFAULT_INITIAL_POTENTIAL,
            "Potencial Final [V]":             DEFAULT_FINAL_POTENTIAL,
            "Potencial Padrão [V]":            DEFAULT_STANDARD_POTENTIAL,
            "Velocidade de Varredura [V.s-1]": DEFAULT_SCAN_RATE,
            "Constante de Equilíbrio":         equilibrium_constant_i,
        }
    )
    currents_equilibrium.append(-solution["Fluxo de O"](solution.t, x=0))
    potentials_equilibrium.append(solution["Potencial Aplicado"](solution.t))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for scan_rate_i, current, potential in zip(scan_rates, currents_scan_rate, potentials_scan_rate):
    ax1.plot(potential, current, label=f"v = {scan_rate_i}")

ax1.set_xlabel(r"$E - E^0$ / V")
ax1.set_ylabel(r"$i$")
ax1.set_xlim([DEFAULT_FINAL_POTENTIAL, DEFAULT_INITIAL_POTENTIAL])
ax1.set_title(r"Efeito de $v$")
ax1.legend(loc="upper left")

for equilibrium_constant_i, current, potential in zip(equilibrium_constants, currents_equilibrium, potentials_equilibrium):
    ax2.plot(potential, current, label=r"$K_{eq}$ = " + str(equilibrium_constant_i))

ax2.set_xlabel(r"$E - E^0$ / V")
ax2.set_ylabel(r"$i$")
ax2.set_xlim([DEFAULT_FINAL_POTENTIAL, DEFAULT_INITIAL_POTENTIAL])
ax2.set_title(r"Efeito de $K_{eq}$")
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()

## Gráfico Interativo:

In [ ]:
interactive_solver = pybamm.ScipySolver()

output = Output()

def parse_float(value):
    """Converte string para float aceitando vírgula ou ponto como separador decimal."""
    return float(str(value).replace(",", "."))

def plot(
    initial_potential_1_str, final_potential_1_str,
    standard_potential_1_str, scan_rate_1_str,
    equilibrium_constant_1_str,
    initial_potential_2_str, final_potential_2_str,
    standard_potential_2_str, scan_rate_2_str,
    equilibrium_constant_2_str,
):
    with output:
        clear_output(wait=True)

        # validação das entradas
        try:
            initial_potential_1    = parse_float(initial_potential_1_str)
            final_potential_1      = parse_float(final_potential_1_str)
            standard_potential_1   = parse_float(standard_potential_1_str)
            scan_rate_1            = parse_float(scan_rate_1_str)
            equilibrium_constant_1 = parse_float(equilibrium_constant_1_str)

            initial_potential_2    = parse_float(initial_potential_2_str)
            final_potential_2      = parse_float(final_potential_2_str)
            standard_potential_2   = parse_float(standard_potential_2_str)
            scan_rate_2            = parse_float(scan_rate_2_str)
            equilibrium_constant_2 = parse_float(equilibrium_constant_2_str)
        except ValueError:
            print("Por favor, insira valores numéricos válidos.")
            return

        if scan_rate_1 == 0 or scan_rate_2 == 0:
            print("A velocidade de varredura não pode ser zero.")
            return

        # ── curva 1 ───────────────────────────────────────────────────────────
        final_time_1 = 2 * (final_potential_1 - initial_potential_1) / scan_rate_1
        solution_1   = interactive_solver.solve(
            model, np.linspace(0, final_time_1, 1000),
            inputs={
                "Potencial Inicial [V]":           initial_potential_1,
                "Potencial Final [V]":             final_potential_1,
                "Potencial Padrão [V]":            standard_potential_1,
                "Velocidade de Varredura [V.s-1]": scan_rate_1,
                "Constante de Equilíbrio":         equilibrium_constant_1,
            }
        )
        current_1   = -solution_1["Fluxo de O"](solution_1.t, x=0)
        potential_1 =  solution_1["Potencial Aplicado"](solution_1.t)

        # ── curva 2 ───────────────────────────────────────────────────────────
        final_time_2 = 2 * (final_potential_2 - initial_potential_2) / scan_rate_2
        solution_2   = interactive_solver.solve(
            model, np.linspace(0, final_time_2, 1000),
            inputs={
                "Potencial Inicial [V]":           initial_potential_2,
                "Potencial Final [V]":             final_potential_2,
                "Potencial Padrão [V]":            standard_potential_2,
                "Velocidade de Varredura [V.s-1]": scan_rate_2,
                "Constante de Equilíbrio":         equilibrium_constant_2,
            }
        )
        current_2   = -solution_2["Fluxo de O"](solution_2.t, x=0)
        potential_2 =  solution_2["Potencial Aplicado"](solution_2.t)

        # ── plotagem ──────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 5))

        ax.plot(potential_1, current_1, color="tab:blue", linewidth=1.5,
                label=f"Curva 1 (v = {scan_rate_1}, $K_{{eq}}$ = {equilibrium_constant_1})")
        ax.plot(potential_2, current_2, color="tab:red",  linewidth=1.5,
                label=f"Curva 2 (v = {scan_rate_2}, $K_{{eq}}$ = {equilibrium_constant_2})")

        ax.set_xlabel(r"$E$ / V")
        ax.set_ylabel(r"$i$")
        ax.set_xlim([min(final_potential_1, final_potential_2), max(initial_potential_1, initial_potential_2)])
        ax.set_title("Comparação de Voltamogramas — Mecanismo EC Reversível")
        ax.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ── Widgets — 1ª Varredura ────────────────────────────────────────────────────

field_initial_potential_1    = Text(value="0.3",   description="Potencial Inicial 1 [V]:",    style={"description_width": "initial"})
field_final_potential_1      = Text(value="-0.3",  description="Potencial Final 1 [V]:",      style={"description_width": "initial"})
field_standard_potential_1   = Text(value="0",     description="Potencial Padrão 1 [V]:",     style={"description_width": "initial"})
field_scan_rate_1            = Text(value="-0.1",  description="Velocidade 1 [V/s]:",         style={"description_width": "initial"})
field_equilibrium_constant_1 = Text(value="0.001", description="Constante de Equilíbrio 1:", style={"description_width": "initial"})


# ── Widgets — 2ª Varredura ────────────────────────────────────────────────────

field_initial_potential_2    = Text(value="0.3",  description="Potencial Inicial 2 [V]:",    style={"description_width": "initial"})
field_final_potential_2      = Text(value="-0.3", description="Potencial Final 2 [V]:",      style={"description_width": "initial"})
field_standard_potential_2   = Text(value="0",    description="Potencial Padrão 2 [V]:",     style={"description_width": "initial"})
field_scan_rate_2            = Text(value="-0.1", description="Velocidade 2 [V/s]:",         style={"description_width": "initial"})
field_equilibrium_constant_2 = Text(value="10",   description="Constante de Equilíbrio 2:", style={"description_width": "initial"})

button = Button(description="Recalcular", button_style="success")

def on_click(b):
    plot(
        field_initial_potential_1.value,    field_final_potential_1.value,
        field_standard_potential_1.value,   field_scan_rate_1.value,
        field_equilibrium_constant_1.value,
        field_initial_potential_2.value,    field_final_potential_2.value,
        field_standard_potential_2.value,   field_scan_rate_2.value,
        field_equilibrium_constant_2.value,
    )

button.on_click(on_click)

interface = VBox([
    HBox([field_initial_potential_1, field_final_potential_1, field_standard_potential_1, field_scan_rate_1, field_equilibrium_constant_1]),
    HBox([field_initial_potential_2, field_final_potential_2, field_standard_potential_2, field_scan_rate_2, field_equilibrium_constant_2]),
    button,
    output,
])
display(interface)

# exibe o gráfico inicial com os valores padrão
plot(
    field_initial_potential_1.value,    field_final_potential_1.value,
    field_standard_potential_1.value,   field_scan_rate_1.value,
    field_equilibrium_constant_1.value,
    field_initial_potential_2.value,    field_final_potential_2.value,
    field_standard_potential_2.value,   field_scan_rate_2.value,
    field_equilibrium_constant_2.value,
)